# 02 - Clean the raw job data

**Goal:** turn the raw Glassdoor file into one tidy table, `data/clean/jobs_clean.csv`.

Run the cells **from top to bottom** with **Shift + Enter**. After each cell, read the printed output and check it looks sensible.
The raw file in `data/raw/` is never modified.

| Step | What we fix |
|---|---|
| 1 | Load the file |
| 2 | Remove duplicate rows |
| 3 | Replace placeholder values (`--`) with proper blanks |
| 4 | Drop rows with no job description |
| 5 | Group job titles into clear roles and remove non-analyst jobs |
| 6 | Standardise cities (Bengaluru, Delhi NCR...) |
| 7 | Convert salary text into numbers (annual INR and LPA) |
| 8 | Work out experience level (Entry / Mid / Senior) |
| 9 | Save the clean file |

## Step 1 - Load the raw file

In [1]:
import re
import glob
import unicodedata
import numpy as np
import pandas as pd

files = glob.glob("../data/raw/*.csv")
print("CSV files found:", files)

RAW_FILE = files[0]   # if you have more than one CSV in data/raw, type the right file path here instead

try:
    df = pd.read_csv(RAW_FILE)
except UnicodeDecodeError:
    df = pd.read_csv(RAW_FILE, encoding="latin-1")

print("Rows loaded:", len(df))
df.head(3)

CSV files found: ['../data/raw/Glassdoor_Job_Postings.csv']
Rows loaded: 900


,company,job_title,company_rating,job_description,location,salary_avg_estimate,salary_estimate_payperiod,company_size,company_founded,employment_type,industry,sector,revenue,career_opportunities_rating,comp_and_benefits_rating,culture_and_values_rating,senior_management_rating,work_life_balance_rating
0,ABB,Junior Data Analyst,4.0,Junior Data Analyst\nTake your next career ste...,Bengaluru,"₹3,25,236",/yr (est.),10000+ Employees,1883,Company - Public,Electronics Manufacturing,Manufacturing,$10+ billion (USD),3.7,3.6,4.0,3.5,3.9
1,Philips,Data Scientist - AI/ML,4.0,Job Title\nData Scientist - AI/ML\nJob Descrip...,Bengaluru,NaN,NaN,10000+ Employees,1891,Company - Public,Healthcare Services & Hospitals,Healthcare,$10+ billion (USD),3.8,3.7,4.0,3.5,4.0
2,HSBC,Data Science GSC’s,3.9,Job description\nGraduate/ Post-graduate degre...,Bengaluru,NaN,NaN,10000+ Employees,1865,Company - Public,Banking & Lending,Finance,$10+ billion (USD),3.6,3.6,3.8,3.4,3.7


## Step 2 - Remove duplicate rows
The same posting can be scraped twice. Counting it twice would inflate every number in the dashboard.

In [2]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Removed {before - len(df)} duplicate rows -> {len(df)} rows left")

Removed 5 duplicate rows -> 895 rows left


## Step 3 - Replace placeholders with real blanks
This dataset uses `--` and `Unknown / Non-Applicable` when a value is missing. We turn them into proper blanks (NaN) and trim stray spaces.

In [3]:
df = df.replace({"--": np.nan, "Unknown / Non-Applicable": np.nan})

for col in df.columns:
    if pd.api.types.is_string_dtype(df[col]) or pd.api.types.is_object_dtype(df[col]):
        df[col] = df[col].str.strip()

df[["company_founded", "industry", "sector", "revenue"]].isnull().sum()

company_founded    329
industry           272
sector             271
revenue            494
dtype: int64

## Step 4 - Drop rows with no job description
We need the description text to find skills in Step 4 of the project, so rows without it are not useful.

In [4]:
before = len(df)
df = df[df["job_description"].notna() & (df["job_description"].str.len() > 30)].reset_index(drop=True)
print(f"Dropped {before - len(df)} rows with no usable description -> {len(df)} rows left")

Dropped 12 rows with no usable description -> 883 rows left


## Step 5 - Group job titles into roles
There are hundreds of different titles. We map each one to a small set of roles. The order of the rules matters: the first rule that matches wins.
Jobs like *Data Entry Operator* and *Data Annotator* are not analyst jobs, so we remove them.

In [5]:
def role_group(title):
    t = str(title).lower()
    if re.search(r"data entry|annotat|typist|back office", t):
        return "Excluded"
    if re.search(r"\bbi\b|business intelligence|power ?bi", t):
        return "BI Analyst"
    if "business analyst" in t:
        return "Business Analyst"
    if "data engineer" in t:
        return "Data Engineer"
    if re.search(r"data scien", t):
        return "Data Scientist"
    if re.search(r"machine learning|\bml\b|\bai\b|deep learning|\bnlp\b", t):
        return "ML/AI Engineer"
    if re.search(r"analyst|analytics", t):
        return "Data Analyst"
    return "Other"

df["role_group"] = df["job_title"].map(role_group)

print(df["role_group"].value_counts())
print()
print("Titles that landed in 'Other' (check these):")
print(df.loc[df["role_group"] == "Other", "job_title"].value_counts().head(15))

role_group
Data Analyst        298
Data Scientist      234
Other               144
Data Engineer       123
Excluded             49
BI Analyst           15
ML/AI Engineer       12
Business Analyst      8
Name: count, dtype: int64

Titles that landed in 'Other' (check these):
job_title
Data Modeler                           3
Data Analysis                          2
ALM Users - Data Mining                2
Clinical Data Coordinator              2
Data Research Associate                2
Data Associate                         2
Data Operations Associate              2
Process Executive - Data               2
Clinical Data Manager                  2
Data Management Executive              2
Lead Associate (Data Analysis)         2
Researcher - Data Operations Intern    1
Clinical Data Coder                    1
Data Coordinator                       1
Employee Data Governance!              1
Name: count, dtype: int64


In [6]:
before = len(df)
df = df[df["role_group"] != "Excluded"].reset_index(drop=True)
print(f"Removed {before - len(df)} non-analyst rows -> {len(df)} rows left")

Removed 49 non-analyst rows -> 834 rows left


## Step 6 - Standardise locations
Problems in the raw data: `Hyderābād` has an accent, `Gurgaon`/`Noida`/`Delhi` are all the Delhi region, `Karnataka` is a state (not a city),
and `Remote` / `India` do not tell us a city. We fix all of these.

In [7]:
def strip_accents(s):
    return "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))

CITY_MAP = {
    "bengaluru": "Bengaluru", "bangalore": "Bengaluru",
    "hyderabad": "Hyderabad", "secunderabad": "Hyderabad",
    "pune": "Pune",
    "mumbai": "Mumbai", "thane": "Mumbai", "navi mumbai": "Mumbai",
    "chennai": "Chennai",
    "delhi": "Delhi NCR", "new delhi": "Delhi NCR", "gurgaon": "Delhi NCR", "gurugram": "Delhi NCR",
    "noida": "Delhi NCR", "greater noida": "Delhi NCR", "ghaziabad": "Delhi NCR", "faridabad": "Delhi NCR",
    "kolkata": "Kolkata", "ahmedabad": "Ahmedabad", "jaipur": "Jaipur",
    "cochin": "Kochi", "kochi": "Kochi", "coimbatore": "Coimbatore",
}

NOT_A_CITY = {
    "india", "remote", "karnataka", "maharashtra", "telangana", "tamil nadu", "gujarat", "kerala",
    "haryana", "uttar pradesh", "west bengal", "rajasthan", "andhra pradesh", "madhya pradesh",
    "punjab", "odisha", "bihar", "goa",
}

def clean_city(loc):
    if not isinstance(loc, str):
        return "Not specified"
    first_part = loc.split(",")[0]
    key = strip_accents(first_part).strip().lower()
    if key in NOT_A_CITY or key == "":
        return "Not specified"
    return CITY_MAP.get(key, strip_accents(first_part).strip().title())

df["city"] = df["location"].map(clean_city)
df["work_mode"] = np.where(df["location"].str.strip().str.lower() == "remote", "Remote", "Office/Other")

print(df["city"].value_counts().head(15))

city
Bengaluru             287
Not specified         192
Delhi NCR              95
Hyderabad              66
Pune                   52
Mumbai                 41
Chennai                36
Ahmedabad               9
Kochi                   8
Jaipur                  6
Thiruvananthapuram      5
Coimbatore              5
Mohali                  5
Chandigarh              4
Gandhinagar             4
Name: count, dtype: int64


## Step 7 - Convert salary text into numbers
Salary looks like `₹3,25,236` (text, with Indian comma style) and comes with a period: per year, per month or per hour.
We remove the symbol and commas, then convert everything to **annual INR**. Monthly x 12. Hourly rows are dropped (there are only a couple).

Important: these are **Glassdoor estimates**, not confirmed offers. We will say so in the README.

In [8]:
def parse_salary(x):
    if not isinstance(x, str):
        return np.nan
    if re.search(r"[-\u2013]|\bto\b", x.lower()):   # a range like 3L-5L: skip, we only trust single values
        return np.nan
    digits = re.sub(r"[^\d.]", "", x)
    return float(digits) if digits else np.nan

def period_multiplier(p):
    if not isinstance(p, str):
        return np.nan
    p = p.lower()
    if "/yr" in p:
        return 1
    if "/mo" in p:
        return 12
    return np.nan   # hourly or unknown

df["salary_number"] = df["salary_avg_estimate"].map(parse_salary)
df["salary_annual_inr"] = df["salary_number"] * df["salary_estimate_payperiod"].map(period_multiplier)
df["salary_lpa"] = (df["salary_annual_inr"] / 100000).round(2)   # LPA = lakhs per annum

print("Rows with a usable salary:", df["salary_lpa"].notna().sum(), "of", len(df))
print()
print(df["salary_lpa"].describe())

Rows with a usable salary: 582 of 834

count    582.000000
mean       6.928763
std        5.521213
min        0.600000
25%        4.210000
50%        5.715000
75%        7.780000
max       52.000000
Name: salary_lpa, dtype: float64


**Sanity check:** the minimum and maximum above should look believable for Indian analyst/data roles (roughly 1 to 60 LPA).
If you see something like 0.01 or 5000, tell me and we will investigate.

## Step 8 - Work out experience level
There is no experience column, so we get it from two places:
1. The **title**: words like *Intern*, *Junior* mean Entry, and *Senior*, *Lead*, *Manager* mean Senior.
2. The **description**: text like `0 - 2 years of experience`. We take the minimum years mentioned near the word "experience".

This is an estimate, not perfect, and we will say so in the case study.

In [9]:
EXP_RE = re.compile(r"(?<!\d)(\d{1,2})\s*(?:-|\u2013|to)?\s*(?:\d{1,2})?\s*\+?\s*(?:years?|yrs?)")

def min_experience(text):
    if not isinstance(text, str):
        return np.nan
    t = text.lower()
    for m in re.finditer(r"experience", t):
        window = t[max(0, m.start() - 80): m.end() + 80]
        found = EXP_RE.search(window)
        if found:
            return int(found.group(1))
    return np.nan

def level_from_title(title):
    t = str(title).lower()
    if re.search(r"intern|trainee|fresher|junior|\bjr\b|graduate|entry", t):
        return "Entry/Fresher"
    if re.search(r"senior|\bsr\b|lead|principal|manager|head|director|architect|staff|\biii\b|\biv\b", t):
        return "Senior (5+ yrs)"
    return None

def final_level(row):
    if pd.notna(row["level_from_title"]):
        return row["level_from_title"]
    y = row["exp_min_years"]
    if pd.isna(y):
        return "Not specified"
    if y <= 1:
        return "Entry/Fresher"
    if y <= 4:
        return "Mid (2-4 yrs)"
    return "Senior (5+ yrs)"

df["exp_min_years"] = df["job_description"].map(min_experience)
df["level_from_title"] = df["job_title"].map(level_from_title)
df["experience_level"] = df.apply(final_level, axis=1)

print(df["experience_level"].value_counts())
print()
print("Minimum years found in descriptions:")
print(df["exp_min_years"].describe())

experience_level
Not specified      265
Mid (2-4 yrs)      255
Entry/Fresher      173
Senior (5+ yrs)    141
Name: count, dtype: int64

Minimum years found in descriptions:
count    515.000000
mean       3.153398
std        3.088475
min        0.000000
25%        2.000000
50%        2.000000
75%        4.000000
max       35.000000
Name: exp_min_years, dtype: float64


## Step 9 - Save the clean file
We keep the useful columns and write the result to `data/clean/jobs_clean.csv`.

In [10]:
keep = [
    "company", "job_title", "role_group", "city", "work_mode", "experience_level", "exp_min_years",
    "salary_annual_inr", "salary_lpa", "company_rating", "company_size", "company_founded",
    "employment_type", "industry", "sector", "revenue", "job_description",
]
clean = df[keep].copy()

clean.to_csv("../data/clean/jobs_clean.csv", index=False, encoding="utf-8")

print("Saved ../data/clean/jobs_clean.csv")
print("Final shape:", clean.shape)
print()
print("Roles:")
print(clean["role_group"].value_counts())
print()
print("Missing values per column:")
print(clean.isnull().sum())

Saved ../data/clean/jobs_clean.csv
Final shape: (834, 17)

Roles:
role_group
Data Analyst        298
Data Scientist      234
Other               144
Data Engineer       123
BI Analyst           15
ML/AI Engineer       12
Business Analyst      8
Name: count, dtype: int64

Missing values per column:
company                0
job_title              0
role_group             0
city                   0
work_mode              0
experience_level       0
exp_min_years        319
salary_annual_inr    252
salary_lpa           252
company_rating       223
company_size         106
company_founded      293
employment_type      106
industry             238
sector               237
revenue              449
job_description        0
dtype: int64


## Checkpoint
Copy the output of the last cell (final shape, roles, missing values) plus the salary and experience summaries and send them back.

In [11]:
import pandas as pd

clean = pd.read_csv("../data/clean/jobs_clean.csv")

print("1) Titles in 'Other':")
print(clean.loc[clean["role_group"] == "Other", "job_title"].value_counts().head(30))

print("\n2) Very low or very high salaries:")
print(clean.loc[(clean["salary_lpa"] < 1.5) | (clean["salary_lpa"] > 30), ["job_title", "salary_lpa"]].sort_values("salary_lpa"))

print("\n3) Experience values above 10 years:")
print(clean.loc[clean["exp_min_years"] > 10, ["job_title", "exp_min_years"]])

1) Titles in 'Other':
job_title
Data Modeler                                   3
Data Analysis                                  2
ALM Users - Data Mining                        2
Clinical Data Coordinator                      2
Data Research Associate                        2
Data Associate                                 2
Data Operations Associate                      2
Process Executive - Data                       2
Clinical Data Manager                          2
Data Management Executive                      2
Lead Associate (Data Analysis)                 2
Researcher - Data Operations Intern            1
Clinical Data Coder                            1
Data Coordinator                               1
Employee Data Governance!                      1
Data Privacy_Consulting                        1
Clinical Data Reviewer                         1
Clinical Data Svs Associate                    1
Clinical Data Coordinator 2                    1
Snowflake Data Modeler               